In [130]:
import pandas as pd

In [131]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [132]:
df=df.drop(columns=['Unnamed: 32']).reset_index(drop=True)
df=df.drop(columns=['id']).reset_index(drop=True)

In [133]:
from sklearn.model_selection import train_test_split

In [134]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

In [135]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [136]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [137]:
from sklearn.preprocessing import LabelEncoder
label=LabelEncoder()
y_train=label.fit_transform(y_train)
y_test=label.transform(y_test)

In [138]:
import torch

In [139]:
X_train_tensor=torch.Tensor(X_train)
X_test_tensor=torch.Tensor(X_test)
y_train_tensor=torch.Tensor(y_train)
y_test_tensor=torch.Tensor(y_test)

In [140]:
df.shape

(569, 31)

In [141]:
X_train_tensor.shape

torch.Size([455, 30])

In [142]:
num_features=30
hidden_layers1=15
hidden_layers2=7
hidden_layers3=3
output_features=2
learning_rate=0.01
# epochs=50

In [143]:
from torch.utils.data import DataLoader,Dataset

In [144]:
from typing import Any


class CustomDataset(Dataset):
    def __init__(self,features,lables):
        self.features=features
        self.labels=lables
        
    def __len__(self):
        return self.features.shape[0]
        
    def __getitem__(self,index):
        return self.features[index],self.labels[index]

        

In [145]:
train_datasets=CustomDataset(X_test_tensor,y_train_tensor)
test_datasets=CustomDataset(X_test_tensor,y_test_tensor)

In [146]:
train_loader=DataLoader(train_datasets,batch_size=32,shuffle=True)
test_loader=DataLoader(test_datasets,batch_size=32,shuffle=True)

In [147]:
import torch.nn as nn

In [148]:
class Model(nn.Module):
    def __init__(self,num_features,hidden_layers1,hidden_layers2,hidden_layers3,output_features):
        super().__init__()
        self.network=nn.Sequential(
            nn.Linear(num_features,hidden_layers1),
            nn.ReLU(),
            nn.Linear(hidden_layers1,hidden_layers2),
            nn.ReLU(),
            nn.Linear(hidden_layers2,hidden_layers3),
            nn.ReLU(),
            nn.Linear(hidden_layers3,output_features),
            
        )
        
    def forward(self,features):
        out=self.network(features)
        return out
    

In [149]:
model = Model(
    num_features=X_train_tensor.shape[1],
    hidden_layers1=15,
    hidden_layers2=7,
    hidden_layers3=3,
    output_features=1
)

In [150]:
model.parameters

<bound method Module.parameters of Model(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=15, bias=True)
    (1): ReLU()
    (2): Linear(in_features=15, out_features=7, bias=True)
    (3): ReLU()
    (4): Linear(in_features=7, out_features=3, bias=True)
    (5): ReLU()
    (6): Linear(in_features=3, out_features=1, bias=True)
  )
)>

In [151]:
from torchinfo import summary
summary(model,input_size=(32,30))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [32, 1]                   --
├─Sequential: 1-1                        [32, 1]                   --
│    └─Linear: 2-1                       [32, 15]                  465
│    └─ReLU: 2-2                         [32, 15]                  --
│    └─Linear: 2-3                       [32, 7]                   112
│    └─ReLU: 2-4                         [32, 7]                   --
│    └─Linear: 2-5                       [32, 3]                   24
│    └─ReLU: 2-6                         [32, 3]                   --
│    └─Linear: 2-7                       [32, 1]                   4
Total params: 605
Trainable params: 605
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.02
Input size (MB): 0.00
Forward/backward pass size (MB): 0.01
Params size (MB): 0.00
Estimated Total Size (MB): 0.01

In [152]:
loss_function=nn.BCEWithLogitsLoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.1)

In [153]:
y_train_tensor = y_train_tensor.long()

epochs=25
for epoch in range(epochs):     
    for batch_features,batch_lables in train_loader:
        #forward pass
        y_pred=model.forward(batch_features)
        
        #loss calculate
        loss=loss_function(y_pred,batch_lables.view(-1,1))
        
        #clear gradient
        optimizer.zero_grad()
        #backward pass
        loss.backward()
        #parameter update
        optimizer.step()
        
    print(f'Epoch:{epoch + 1 },Loss:{loss.item()}')


Epoch:1,Loss:0.6890122890472412
Epoch:2,Loss:0.6947173476219177
Epoch:3,Loss:0.6940367221832275
Epoch:4,Loss:0.6833306550979614
Epoch:5,Loss:0.6428782939910889
Epoch:6,Loss:0.6626861095428467
Epoch:7,Loss:0.6871028542518616
Epoch:8,Loss:0.707187831401825
Epoch:9,Loss:0.6501117944717407
Epoch:10,Loss:0.6463126540184021
Epoch:11,Loss:0.6226765513420105
Epoch:12,Loss:0.6663616895675659
Epoch:13,Loss:0.6947348713874817
Epoch:14,Loss:0.5624269843101501
Epoch:15,Loss:0.6396596431732178
Epoch:16,Loss:0.6672292351722717
Epoch:17,Loss:0.6398941874504089
Epoch:18,Loss:0.513722836971283
Epoch:19,Loss:0.6039004921913147
Epoch:20,Loss:0.7066084146499634
Epoch:21,Loss:0.6358566284179688
Epoch:22,Loss:0.6727249622344971
Epoch:23,Loss:0.6346456408500671
Epoch:24,Loss:0.6355021595954895
Epoch:25,Loss:0.5315937995910645


In [154]:
# import matplotlib.pyplot as plt 

# plt.plot(range(epochs),loss)
# plt.xlabel('epoch')
# plt.ylabel('loss')
# plt.show()

In [158]:
model.eval()
accuracy_list=[]

with torch.no_grad(): # disable gradient tracking 
    for batch_features,batch_lables in test_loader:
        
        #forward pass
        y_pred=model(batch_features)
        y_pred=(y_pred > 0.8).float()
        
        batch_accuracy=(y_pred.view(-1)==batch_lables).float().mean().item()
        accuracy_list.append(batch_accuracy)
        
overall_accuracy=sum(accuracy_list)/ len(accuracy_list)
print(f'Accuracy:{overall_accuracy:.4f}')

Accuracy:0.6033
